In [ ]:
import sys
import numpy as np

from pathlib import Path

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.graph_stats as graph_stats
import utils.graph_utils as graph_utils
import utils.vis as vis

from config_handlers.rk_handler import RKHandler

# Main

In [ ]:
# Load RK config
rk_hdl = RKHandler()
MODELS = rk_hdl.get_model_list()
""" Skipping Qwen2.5-VL for now. """
MODELS.remove("qwen2_5_vl")
rk_hdl.set_curr_model(MODELS[0])
DATASETS = rk_hdl.get_ds_list()
base_dir = Path("..")

all_datasets = DATASETS * 4
all_models = MODELS * 4
all_models.sort()

## Exploring differences between RKs with greedy and sampling-based decoding

In this notebook, we load the `parsed` RK instead onf the `final` ones because we do not need the bounding boxes data.

In [ ]:
def get_valid_rks(all_models: list[str], all_datasets: list[str], version: int) -> dict:
    all_rks = {}

    for model, ds in zip(all_models, all_datasets):
        rk_hdl.set_curr_model(model)
        rk_hdl.set_curr_ds(ds)
        parsed_rk_path = base_dir.joinpath(
            str(rk_hdl.get_rk_parsed_path()).format(version)
        )
        parsed_rks = data_io.load_jsonl(parsed_rk_path)

        # Set up dict for current model x dataset combination
        if model not in all_rks:
            all_rks[model] = {}
        if ds not in all_rks[model]:
            all_rks[model][ds] = {}
        rk_dict = {}

        # Create entry for each sample
        for curr_sample in parsed_rks:
            question_id = curr_sample["question_id"]
            # Skip if there is any structured triple
            if len(curr_sample["triple_objs"]) == 0:
                # print(f"No RK found for {question_id} ({model} x {ds})")
                continue

            # Otherwise, record entry
            if question_id not in rk_dict:
                rk_dict[question_id] = curr_sample["triple_objs"]

        # Save for later
        all_rks[model][ds] = rk_dict

    return all_rks

In [ ]:
def print_rk_details(data_det: dict, data_sam: dict) -> None:
    for model in MODELS:
        print("=" * 10, f"{model.upper()}", "=" * 10)
        for ds in DATASETS:
            print("-" * 50)
            print(f"{ds.upper()}")
            print("-" * 50)
            print("\t\t = Greedy = \t = Nucleus =")

            # Get the number of entries
            n_samples_det = len(data_det["sample_stats"][model][ds])
            n_samples_sam = len(data_sam["sample_stats"][model][ds])
            print(f"Entries \t {n_samples_det} \t\t {n_samples_sam}")
            print("-" * 50)

            # Get the number of concepts extracted
            n_concepts_det = data_det["summaries"][model][ds]["count_concepts"]
            n_concepts_sam = data_sam["summaries"][model][ds]["count_concepts"]
            avg_c_det = data_det["summaries"][model][ds]["avg_concepts"]
            avg_c_sam = data_sam["summaries"][model][ds]["avg_concepts"]
            std_c_det = data_det["summaries"][model][ds]["std_concepts"]
            std_c_sam = data_sam["summaries"][model][ds]["std_concepts"]
            print(f"N. Nodes \t {n_concepts_det} \t\t {n_concepts_sam}")
            print(f"Avg. Concepts \t {avg_c_det:.2f} \t\t {avg_c_sam:.2f}")
            print(f"Std. Concepts \t {std_c_det:.2f} \t\t {std_c_sam:.2f}")
            print("-" * 50)

            # Get the number of relations extracted
            n_rels_det = data_det["summaries"][model][ds]["count_preds"]
            n_rels_sam = data_sam["summaries"][model][ds]["count_preds"]
            avg_p_det = data_det["summaries"][model][ds]["avg_preds"]
            avg_p_sam = data_sam["summaries"][model][ds]["avg_preds"]
            std_p_det = data_det["summaries"][model][ds]["std_preds"]
            std_p_sam = data_sam["summaries"][model][ds]["std_preds"]
            print(f"N. RKs \t\t {n_rels_det} \t\t {n_rels_sam}")
            print(f"Avg. Preds \t {avg_p_det:.2f} \t\t {avg_p_sam:.2f}")
            print(f"Std. Preds \t {std_p_det:.2f} \t\t {std_p_sam:.2f}")
            print("-" * 50)

            # Compare IC and Entropy at the graph level
            avg_ic_det = data_det["summaries"][model][ds]["avg_ic"]
            std_ic_det = data_det["summaries"][model][ds]["std_ic"]
            avg_ic_sam = data_sam["summaries"][model][ds]["avg_ic"]
            std_ic_sam = data_sam["summaries"][model][ds]["std_ic"]

            print(f"Avg. IC \t {avg_ic_det:.2f} \t\t {avg_ic_sam:.2f}")
            print(f"Std. IC \t {std_ic_det:.2f} \t\t {std_ic_sam:.2f}")

Loading dictionary from VG1800 with concept, predicates, and their counts and frequencies

In [ ]:
# Loading concept and predicate frequencies from VG1800 dataset from IETrans.
vg1800_dict = graph_utils.load_vg1800_dict()

In [ ]:
def get_rk_details(rks: dict) -> dict:
    data = {
        "graphs": {model: {} for model in MODELS},
        "summaries": {model: {} for model in MODELS},
        "sample_stats": {model: {} for model in MODELS},
        "count_concepts": {model: {} for model in MODELS},
        "count_preds": {model: {} for model in MODELS},
    }

    for model, ds in zip(all_models, all_datasets):
        print(f"Processing {model} -- {ds}")
        rk_hdl.set_curr_model(model)
        rk_hdl.set_curr_ds(ds)
        if ds not in data["graphs"][model]:
            data["graphs"][model][ds] = {}

        # Create NX graphs for each data sample
        for q_idx, q_data in rks[model][ds].items():
            data["graphs"][model][ds][q_idx] = graph_utils.simple_rk_to_nx(q_data)

        sample_stats, stats_summary = graph_stats.compute_stats(
            data["graphs"][model][ds], vg1800_dict
        )

        # Update
        data["sample_stats"][model][ds] = sample_stats
        data["summaries"][model][ds] = stats_summary
        data["count_concepts"][model][ds] = stats_summary["count_concepts"]
        data["count_preds"][model][ds] = stats_summary["count_preds"]

    return data

Greedy decoding (v4)

In [ ]:
prompt_version = 4

# Load RK data
rks_det = get_valid_rks(all_models, all_datasets, prompt_version)
# Process RK data
data_det = get_rk_details(rks_det)

Nucleus sampling (v5)

In [ ]:
prompt_version = 5

# Load RK data
rks_sam = get_valid_rks(all_models, all_datasets, prompt_version)
# Process RK data
data_sam = get_rk_details(rks_sam)

Comparison

In [ ]:
print_rk_details(data_det, data_sam)

# Save results to disk

Save statistics

In [ ]:
# Create output dir for statistics
out_dir_det = Path(".", "raw_stats", "rk_det")
data_io.make_dir(out_dir_det)
out_dir_sam = Path(".", "raw_stats", "rk_sam")
data_io.make_dir(out_dir_sam)

for model, ds in zip(all_models, all_datasets):
    print(f"Saving {model} -- {ds}")

    # Save statistics greedy decoding
    ## Summaries
    summary_det_file = out_dir_det.joinpath(model, ds, "summary.json")
    data_io.make_dir(summary_det_file.parent)
    data_io.save_json(data_det["summaries"][model][ds], summary_det_file)
    ## Sample-level
    sample_def_file = out_dir_det.joinpath(model, ds, "samples.json")
    data_io.save_json(data_det["sample_stats"][model][ds], sample_def_file)

    # Save statistics nucleus sampling decoding
    ## Summaries
    summary_sam_file = out_dir_sam.joinpath(model, ds, "summary.json")
    data_io.make_dir(summary_sam_file.parent)
    data_io.save_json(data_sam["summaries"][model][ds], summary_sam_file)
    # Sample-level
    sample_sam_file = out_dir_sam.joinpath(model, ds, "samples.json")
    data_io.save_json(data_sam["sample_stats"][model][ds], sample_sam_file)

Save statistics graphs

In [ ]:
# Create output dir for charts
charts_dir = Path(".", "stats_charts")
data_io.make_dir(charts_dir)

In [ ]:
def prep_data_plots(data: dict) -> tuple[dict, dict]:
    all_count_concepts = {}
    all_count_preds = {}

    for model in MODELS:
        rk_hdl.set_curr_model(model)
        all_count_concepts[model] = {}
        all_count_preds[model] = {}
        for ds in rk_hdl.get_ds_list():
            rk_hdl.set_curr_ds(ds)
            all_count_concepts[model][ds] = [
                v["count_concepts"] for v in data[model][ds].values()
            ]
            all_count_preds[model][ds] = [
                v["count_preds"] for v in data[model][ds].values()
            ]
    return all_count_concepts, all_count_preds

In [ ]:
# Greedy decoding
all_count_concepts, all_count_preds = prep_data_plots(data_det["sample_stats"])
vis.plot_rk_rels(all_count_preds, charts_dir.joinpath("box_rk_det.pdf"))
vis.plot_rk(
    all_count_concepts, all_count_preds, charts_dir.joinpath("box_rk_det_full.pdf")
)

In [ ]:
# Nucleus sampling
all_count_concepts, all_count_preds = prep_data_plots(data_sam["sample_stats"])
vis.plot_rk_rels(all_count_preds, charts_dir.joinpath("box_rk_sam.pdf"))
vis.plot_rk(
    all_count_concepts, all_count_preds, charts_dir.joinpath("box_rk_sam_full.pdf")
)

# Checking final RKs

In [ ]:
def get_final_rks(all_models: list[str], all_datasets: list[str]) -> dict:
    all_rks = {}

    for model, ds in zip(all_models, all_datasets):
        rk_hdl.set_curr_model(model)
        rk_hdl.set_curr_ds(ds)
        parsed_rk_path = base_dir.joinpath(rk_hdl.get_rk_final_path())
        parsed_rks = data_io.load_jsonl(parsed_rk_path)

        # Set up dict for current model x dataset combination
        if model not in all_rks:
            all_rks[model] = {}
        if ds not in all_rks[model]:
            all_rks[model][ds] = {}
        rk_dict = {}

        # Create entry for each sample
        for curr_sample in parsed_rks:
            question_id = curr_sample["question_id"]
            # Skip if there is any structured triple
            if len(curr_sample["triple_objs"]) == 0:
                # print(f"No RK found for {question_id} ({model} x {ds})")
                continue

            # Otherwise, parse and record entry
            if question_id not in rk_dict:
                # The final RKs also have the bounding boxes info. Clean these up.
                simple_triples = []
                for triple in curr_sample["triple_objs"]:
                    simple_triples.append(
                        {
                            "from_concept": triple["from_concept"]["bb_label"][
                                "bb_label_text"
                            ],
                            "relationship": triple["rel_label"]["rel_label_text"],
                            "to_concept": triple["to_concept"]["bb_label"][
                                "bb_label_text"
                            ],
                        }
                    )
                rk_dict[question_id] = simple_triples

        # Save for later
        all_rks[model][ds] = rk_dict

    return all_rks


def print_rk_final_details(data: dict) -> None:
    for model in MODELS:
        print("=" * 10, f"{model.upper()}", "=" * 10)
        for ds in DATASETS:
            print("-" * 50)
            print(f"{ds.upper()}")
            print("-" * 50)
            curr = data["sample_stats"][model][ds]

            # Get the number of entries
            n_samples = len(data["sample_stats"][model][ds])
            print(f"Entries \t {n_samples}")
            print("-" * 50)

            # Get the number of concepts extracted
            n_concepts = data["summaries"][model][ds]["count_concepts"]
            avg_c = data["summaries"][model][ds]["avg_concepts"]
            std_c = data["summaries"][model][ds]["std_concepts"]
            counts_c = [sample["count_concepts"] for sample in curr.values()]
            q1_c = np.quantile(counts_c, 0.25).item()
            q3_c = np.quantile(counts_c, 0.75).item()
            # disp_c = (q3_c - q1_c) / (q3_c + q1_c)
            median_c = np.median(counts_c).item()
            iqr_c = q3_c - q1_c
            print(f"N. Nodes \t {n_concepts}")
            print(f"Median concepts: \t {median_c:.2f}")
            print(f"IQR concepts: \t {iqr_c:.2f} ({q1_c}, {q3_c})")
            print(f"Avg. Concepts \t {avg_c:.2f}")
            print(f"Std. Concepts \t {std_c:.2f}")
            print("-" * 50)

            # Get the number of relations extracted
            n_rels = data["summaries"][model][ds]["count_preds"]
            avg_p = data["summaries"][model][ds]["avg_preds"]
            std_p = data["summaries"][model][ds]["std_preds"]
            counts_p = [sample["count_preds"] for sample in curr.values()]
            q1_p = np.quantile(counts_p, 0.25).item()
            q3_p = np.quantile(counts_p, 0.75).item()
            # disp_p = (q3_p - q1_p) / (q3_p + q1_p)
            median_p = np.median(counts_p).item()
            iqr_p = q3_p - q1_p
            print(f"N. RKs \t\t {n_rels}")
            print(f"Median predicates: \t {median_p:.2f}")
            print(f"IQR predicates: \t {iqr_p:.2f} ({q1_p}, {q3_p})")
            print(f"Avg. Preds \t {avg_p:.2f}")
            print(f"Std. Preds \t {std_p:.2f}")
            print("-" * 50)

            # Compare IC and Entropy at the graph level
            # avg_ic_det = data_det["summaries"][model][ds]["avg_ic"]
            # std_ic_det = data_det["summaries"][model][ds]["std_ic"]
            # print(f"Avg. IC \t {avg_ic_det:.2f}")
            # print(f"Std. IC \t {std_ic_det:.2f}")

In [ ]:
# Load RK config
MODELS = rk_hdl.get_model_list()
rk_hdl.set_curr_model(MODELS[0])
DATASETS = rk_hdl.get_ds_list()
all_datasets = DATASETS * 4
all_models = MODELS * 4
all_models.sort()

# Load RK data
rks_det = get_final_rks(all_models, all_datasets)
# Process RK data
data_det = get_rk_details(rks_det)
print_rk_final_details(data_det)

In [ ]:
# Create output dir for statistics
out_dir_det = Path(".", "raw_stats", "rk_final")
data_io.make_dir(out_dir_det)

for model, ds in zip(all_models, all_datasets):
    print(f"Saving {model} -- {ds}")

    ## Summaries
    summary_det_file = out_dir_det.joinpath(model, ds, "summary.json")
    data_io.make_dir(summary_det_file.parent)
    data_io.save_json(data_det["summaries"][model][ds], summary_det_file)
    ## Sample-level
    sample_def_file = out_dir_det.joinpath(model, ds, "samples.json")
    data_io.save_json(data_det["sample_stats"][model][ds], sample_def_file)

In [ ]:
all_count_concepts, all_count_preds = prep_data_plots(data_det["sample_stats"])
vis.plot_rk_rels(all_count_preds, charts_dir.joinpath("box_rk_final.pdf"))
vis.plot_rk(
    all_count_concepts, all_count_preds, charts_dir.joinpath("box_rk_final_full.pdf")
)